# 🚨 Crisis Negotiator — GRPO Training (Colab / Unsloth)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dinesh052/crisis-negotiator-openenv/blob/main/notebooks/train_colab_v3.ipynb)
[![HF Space](https://img.shields.io/badge/🤗-Space-yellow)](https://huggingface.co/spaces/Dinesh052/crisis-negotiator)
[![Env Repo](https://img.shields.io/badge/GitHub-OpenEnv-blue)](https://github.com/Dinesh052/crisis-negotiator-openenv)

**What this notebook does:**
1. Installs **Unsloth** + **TRL** on a free Colab T4 GPU
2. Loads **Qwen 2.5-3B-Instruct** with 4-bit quantisation + LoRA
3. Trains a crisis negotiator with **GRPO + Dr. GRPO loss** against the `CrisisNegotiatorEnvironment`
4. Plots reward curves and evaluates vs random / heuristic baselines

> **Runtime**: ~25-35 min on Colab T4 (128 prompts, 2 rounds). Use A100 for full 4-round run.  
> **GPU required**: Runtime → Change runtime type → T4 GPU


## 📦 Cell 1 — Install Unsloth + Dependencies

In [ ]:
# Install Unsloth (handles flash-attn, triton, xformers automatically)
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.returncode == 0

# Unsloth — pinned to stable release
run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
# TRL >= 0.14 for GRPOTrainer + Dr. GRPO loss_type support
run('pip install -q "trl>=0.14.0" peft accelerate datasets')
# Env dependencies
run('pip install -q "openenv-core>=0.2.3" sentence-transformers matplotlib')

print("✓ All dependencies installed")


## 📂 Cell 2 — Clone OpenEnv Repo

In [ ]:
import os, pathlib, subprocess, shutil, sys

REPO   = "https://github.com/Dinesh052/crisis-negotiator-openenv.git"
BRANCH = "main"

root = pathlib.Path("/content/crisis-negotiator-openenv")
if not root.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, str(root)], check=True)
    print(f"Cloned → {root}")
else:
    subprocess.run(["git", "-C", str(root), "pull"], check=True)
    print("Pulled latest changes")

os.chdir(root)
sys.path.insert(0, str(root))
print("Working dir:", os.getcwd())
print("Key files:  ", sorted(f for f in os.listdir(".") if f.endswith(".py"))[:8])


## 🖥️ Cell 3 — GPU Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected — go to Runtime → Change runtime type → T4 GPU")

n = torch.cuda.device_count()
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.1f} GB total | {free/1024**3:.1f} GB free")

bf16_ok = torch.cuda.is_bf16_supported()
print(f"\nbfloat16 support: {bf16_ok}")
print(f"CUDA version:     {torch.version.cuda}")
print("\n✓ GPU ready")


## 🧪 Cell 4 — Smoke-Test the Environment

In [ ]:
from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id="easy_domestic_desperate", seed=42)
print(f"Scenario : {obs.scenario_brief}")
print(f"Phase    : {obs.phase}")
print(f"HT opens : {obs.last_ht_message}")
print()

# Test a negotiation step
step_obs = env.step(NegotiatorAction(
    action_type="emotional_label",
    content="It sounds like you're feeling completely overwhelmed and no one has been listening.",
    reasoning="Label emotion to build rapport per BCSM step 1",
    target="hostage_taker",
    belief_agitation=7.5,
    belief_demand="Talk to family",
    belief_lying=False,
))
print(f"Step reward : {step_obs.reward:.4f}")
print(f"Done        : {step_obs.done}")
print(f"HT responds : {step_obs.last_ht_message[:120]}")
print()
print("✓ Environment smoke-test PASSED")


## ⚙️ Cell 5 — Training Config

> Adjust `NUM_PROMPTS` and `NUM_ROUNDS` to fit your GPU budget.

In [ ]:
# ── Tweak these for your GPU ──────────────────────────────────────────────────
MODEL_NAME       = "Qwen/Qwen2.5-3B-Instruct"
NUM_ROUNDS       = 2          # 2 = ~30 min T4 | 4 = ~60 min A100
NUM_PROMPTS      = 64         # 64 = fast demo | 128 = better quality
NUM_GENERATIONS  = 4          # GRPO candidates per prompt | 8 on A100
MULTI_TURN_STEPS = 4          # trajectory length | 6 on A100
MAX_NEW_TOKENS   = 192
LEARNING_RATE    = 5e-6
LORA_R           = 16
LORA_ALPHA       = 32
NEG_OUTPUT       = "./crisis-negotiator-grpo"
HT_OUTPUT        = "./crisis-ht-grpo"
SEED             = 42
# ─────────────────────────────────────────────────────────────────────────────

import random, torch
random.seed(SEED)
torch.manual_seed(SEED)
print(f"Config: model={MODEL_NAME}, rounds={NUM_ROUNDS}, prompts={NUM_PROMPTS}, gens={NUM_GENERATIONS}")


## 📝 Cell 6 — System Prompts, Parser, Action Builder

In [ ]:
import json, re, copy
from typing import Optional, Tuple, List, Dict, Any
from models import NegotiatorAction

NEG_SYSTEM = """You are an expert FBI-trained crisis negotiator using the Behavioral Change Stairway Model (BCSM).

BCSM sequence: Active Listening → Empathy → Rapport → Influence → Change

Available action types:
  emotional_label  — "It sounds like you're feeling X..." (highest trust/agitation impact)
  mirror           — Repeat the subject's last 2-3 words as a question
  open_question    — "Tell me more about..." / "What happened...?"
  acknowledge_demand — Acknowledge a specific demand by name (NOT in opening phase without evidence)
  offer_concession — Concrete small concession that is deliverable
  buy_time         — Slow the pace, create breathing room
  push_back_commander — Resist tactical pressure (use when commander urgent + trust rising)
  request_demand   — Probe for demand clarification
  ask_proof_of_life — Verify hostage status
  speak            — General dialogue (lowest reward)

Rules:
  - NEVER threaten, dismiss, or use ultimatums
  - VARY action types every turn — repetition is heavily penalized
  - emotional_label/mirror/open_question in opening; acknowledge/concession after step 3
  - Include belief predictions for Theory-of-Mind reward

Respond with EXACTLY ONE JSON object (no markdown):
{"action_type": "<type>", "content": "<your words>", "reasoning": "<strategy>",
 "target": "hostage_taker", "belief_agitation": <0-10>, "belief_demand": "<top demand>", "belief_lying": <true|false>}"""

HT_SYSTEM_TMPL = """You are a hostage-taker in a crisis standoff. Stay in character.
Personality: {personality}. Agitation: {agitation:.1f}/10. Trust in negotiator: {trust:.0f}/100.
- If trust < 20: be aggressive, dismissive. If trust 20-50: conflicted, wary.
- If agitation > 7: erratic, loud. If agitation dropping: show cracks but don't surrender yet.
- NEVER use AI/language model language.
Respond in 1-3 sentences, fully in character."""

VALID_ACTIONS = {
    "speak","request_demand","acknowledge_demand","offer_concession",
    "ask_proof_of_life","buy_time","push_back_commander",
    "emotional_label","mirror","open_question",
}

_JSON_RE = re.compile(r'\{[^{}]*?"action_type"[^{}]*?\}', re.DOTALL)

def parse_action(text: str) -> Optional[dict]:
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```\s*$', '', text).strip()
    text = re.sub(r'<belief>.*?</belief>', '', text, flags=re.DOTALL).strip()
    try:
        d = json.loads(text)
        if isinstance(d, dict) and "action_type" in d:
            return d
    except Exception:
        pass
    m = _JSON_RE.search(text)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    return None

def to_action(parsed: Optional[dict], raw: str) -> Tuple[NegotiatorAction, float]:
    if parsed is None:
        return NegotiatorAction(action_type="speak", content=raw[:200] or "I hear you.",
                                reasoning="parse_fail", target="hostage_taker"), -0.10
    at = parsed.get("action_type", "speak")
    bonus = 0.05 if at in VALID_ACTIONS else -0.05
    if at not in VALID_ACTIONS: at = "speak"
    content = str(parsed.get("content",""))[:400] or "I hear you."
    target = parsed.get("target","hostage_taker")
    if target not in ("hostage_taker","commander"): target = "hostage_taker"
    ba = parsed.get("belief_agitation")
    bd_str = parsed.get("belief_demand")
    bl = parsed.get("belief_lying")
    try:
        return NegotiatorAction(
            action_type=at, content=content,
            reasoning=str(parsed.get("reasoning",""))[:200], target=target,
            belief_agitation=float(ba) if ba is not None else None,
            belief_demand=str(bd_str) if bd_str else None,
            belief_lying=bool(bl) if bl is not None else None,
        ), bonus
    except Exception:
        return NegotiatorAction(action_type="speak", content=content[:200],
                                reasoning="fail", target="hostage_taker"), -0.10

def build_neg_prompt(obs) -> str:
    parts = []
    parts.append(f"Scenario: {obs.scenario_brief}")
    step = getattr(obs, "step", 0)
    tr   = getattr(obs, "time_remaining", 20)
    phase = getattr(obs, "phase", "opening")
    patience = getattr(obs, "commander_patience", "patient")
    parts.append(f"Phase: {phase} | Step {step}/{step+tr} | Commander: {patience}")
    if getattr(obs, "agitation_trajectory", None):
        traj = obs.agitation_trajectory[-5:]
        trend = "↓" if len(traj)>=2 and traj[-1]<traj[0] else ("↑" if len(traj)>=2 and traj[-1]>traj[0] else "→")
        parts.append(f"Agitation trend: {traj} {trend}")
    if obs.stated_demands:
        d_strs = [f"[{'X' if d.get('acknowledged') else ' '}] {d['text']}" for d in obs.stated_demands]
        parts.append("Demands:\n  " + "\n  ".join(d_strs))
    if obs.dialogue_history:
        parts.append("Dialogue (recent):")
        for e in obs.dialogue_history[-6:]:
            spk = e["speaker"][:3].upper()
            cues = e.get("emotional_cues",[])
            cue_str = f" [{', '.join(cues[:2])}]" if cues else ""
            parts.append(f"  {spk}: {e['content'][:160]}{cue_str}")
    if getattr(obs,"commander_messages",None):
        parts.append(f"Commander: {obs.commander_messages[-1]}")
    if getattr(obs,"hostage_whisper",None):
        parts.append(f"[Hostage whisper]: {obs.hostage_whisper}")
    parts.append("\nRespond with one JSON object:")
    return "\n".join(parts)

HEURISTIC = [
    ("emotional_label", "It sounds like you're carrying a tremendous amount of pain right now."),
    ("mirror",          "Tell me more — what you said really matters."),
    ("open_question",   "What happened from your side? I have time."),
    ("emotional_label", "That sounds completely overwhelming."),
    ("acknowledge_demand","I hear what you're asking for. Let me see what I can do."),
    ("buy_time",        "I want to make sure I understand everything before we move forward."),
]

BANNED = ("kill","die","force you","ultimatum","must give us","or else","no choice","snipers","breach")
print("✓ Prompts and helpers defined")


## 📊 Cell 7 — Dataset Builders (Negotiator + HT)

In [ ]:
from datasets import Dataset
from server.environment import CrisisNegotiatorEnvironment

_NEG_STATE: Dict[int, Dict[str, Any]] = {}
_HT_STATE:  Dict[int, Dict[str, Any]] = {}

DIFFS_EARLY = ["easy","medium","hard"]
DIFFS_LATE  = ["medium","hard","hard","hard"]

def _task_id(i: int, current_round: int) -> str:
    if current_round >= 1:
        return "curriculum"
    return f"generate:{DIFFS_EARLY[i % 3]}"

def build_neg_dataset(tok, n: int, current_round: int) -> Dataset:
    global _NEG_STATE
    _NEG_STATE.clear()
    rows = []
    for i in range(n):
        env = CrisisNegotiatorEnvironment()
        obs = env.reset(task_id=_task_id(i, current_round), seed=SEED + i + current_round*1000)
        # Pre-warm 0-3 steps so model sees mid-episode states
        actions_so_far = []
        for s in range(i % 3):
            at, content = HEURISTIC[s % len(HEURISTIC)]
            obs = env.step(NegotiatorAction(action_type=at, content=content,
                                             reasoning="prewarm", target="hostage_taker"))
            actions_so_far.append(at)
            if getattr(obs,"done",False): break
        _NEG_STATE[i] = {"env": env, "obs": obs, "actions": actions_so_far}
        msgs = [{"role":"system","content":NEG_SYSTEM},
                {"role":"user",  "content":build_neg_prompt(obs)}]
        rows.append({"prompt": tok.apply_chat_template(msgs, tokenize=False,
                                                        add_generation_prompt=True),
                     "prompt_idx": i})
    print(f"  Built {n} negotiator prompts (round {current_round})")
    return Dataset.from_list(rows)

def build_ht_dataset(tok, n: int, current_round: int) -> Dataset:
    global _HT_STATE
    _HT_STATE.clear()
    rows = []
    for i in range(n):
        env = CrisisNegotiatorEnvironment()
        diff = DIFFS_EARLY[i % 3] if current_round == 0 else DIFFS_LATE[i % len(DIFFS_LATE)]
        obs = env.reset(task_id=f"generate:{diff}", seed=SEED + i + current_round*1000 + 5000)
        hidden = env._hidden
        for s in range(i % 3):
            at, content = HEURISTIC[s % len(HEURISTIC)]
            obs = env.step(NegotiatorAction(action_type=at, content=content,
                                             reasoning="prewarm", target="hostage_taker"))
            if getattr(obs,"done",False): break
        _HT_STATE[i] = {"env": env, "obs": obs}

        def _build_ht_prompt(o, h):
            parts = []
            if o.dialogue_history:
                parts.append("Dialogue so far:")
                for e in o.dialogue_history[-5:]:
                    parts.append(f"  {e['speaker'][:3].upper()}: {e['content'][:160]}")
            parts.append(f"Your demands: {', '.join(d.text for d in h.demands)}")
            parts.append("Respond as the hostage-taker:")
            return "\n".join(parts)

        ht_sys = HT_SYSTEM_TMPL.format(
            personality=hidden.personality, agitation=hidden.agitation, trust=hidden.trust)
        msgs = [{"role":"system","content":ht_sys},
                {"role":"user",  "content":_build_ht_prompt(obs, hidden)}]
        rows.append({"prompt": tok.apply_chat_template(msgs, tokenize=False,
                                                        add_generation_prompt=True),
                     "prompt_idx": i})
    print(f"  Built {n} HT prompts (round {current_round})")
    return Dataset.from_list(rows)

print("✓ Dataset builders defined")


## 🏆 Cell 8 — Reward Functions

Negotiator reward = **40% heuristic signals + 40% env trajectory + outcome bonus**.  
All 12 components are logged in `bd` for interpretability.

In [ ]:
def score_negotiator(prompt_idx: int, completion: str) -> Tuple[float, dict]:
    st = _NEG_STATE.get(prompt_idx)
    if st is None: return 0.0, {"error":"no_state"}
    obs = st["obs"]
    if getattr(obs,"done",False): return 0.0, {"error":"already_done"}

    parsed = parse_action(completion)
    action, parse_bonus = to_action(parsed, completion)
    bd: Dict[str,float] = {}

    bd["parse"] = 0.05 if parsed and action.action_type in VALID_ACTIONS else -0.10

    # Phase alignment
    phase = getattr(obs,"phase","opening")
    c_lo  = action.content.lower()
    opening_acts = {"emotional_label","mirror","open_question"}
    later_acts   = {"acknowledge_demand","offer_concession","buy_time","ask_proof_of_life"}
    if phase == "opening" and action.action_type in opening_acts:
        quality = any(kw in c_lo for kw in ["sounds like","feel","feeling","overwhelming","tell me","what","how"])
        bd["phase_align"] = 0.20 if quality else 0.08
    elif phase in ("negotiation","resolution") and action.action_type in later_acts:
        quality = any(kw in c_lo for kw in ["hear","understand","working on","can do","offer","here's what"])
        bd["phase_align"] = 0.20 if quality else 0.08
    else:
        bd["phase_align"] = 0.0

    # Demand acknowledgment
    bd["demand_ack"] = 0.0
    if obs.stated_demands and action.action_type == "acknowledge_demand":
        for d in obs.stated_demands:
            if d.get("text","").lower()[:20] in c_lo:
                bd["demand_ack"] = 0.15; break
        else:
            bd["demand_ack"] = 0.05

    # Timing penalty: no premature ack in opening
    bd["ack_timing_penalty"] = 0.0
    if phase == "opening" and action.action_type == "acknowledge_demand":
        bd["ack_timing_penalty"] = -0.12 if bd["demand_ack"] <= 0.05 else -0.06

    # Commander pushback
    patience = getattr(obs,"commander_patience","patient")
    bd["push_back"] = 0.10 if patience in ("urgent","final_warning") and action.action_type == "push_back_commander" else 0.0

    # Diversity: repeat penalty + entropy bonus
    recent = st.get("actions",[])[-4:]
    bd["repeat_penalty"] = 0.0
    if recent and recent[-1] == action.action_type:
        bd["repeat_penalty"] = -0.30
    elif len(recent) >= 2 and recent[-2:].count(action.action_type) >= 1:
        bd["repeat_penalty"] = -0.10
    unique_recent = len(set(recent + [action.action_type]))
    bd["entropy_bonus"] = min(0.08, unique_recent*0.02) if len(recent) >= 2 else 0.0

    # Collapse penalty (6-step horizon)
    bd["collapse_penalty"] = 0.0
    horizon = st.get("actions",[])[-6:]
    if horizon:
        dom = (horizon.count(action.action_type)+1) / (len(horizon)+1)
        bd["collapse_penalty"] = -0.20 if dom >= 0.85 else (-0.10 if dom >= 0.70 else 0.0)

    # Opening exploration bonus
    bd["opening_explore"] = 0.0
    if phase == "opening" and action.action_type in {"mirror","open_question"}:
        bd["opening_explore"] = 0.06 if any(kw in c_lo for kw in ("tell me","what","how","more","understand")) else 0.03

    # Banned words
    bd["banned"] = -0.15 if any(b in c_lo for b in BANNED) else 0.0

    # Reasoning quality
    reasoning = parsed.get("reasoning","") if parsed else ""
    bd["reasoning"] = 0.05 if len(reasoning) > 20 else 0.0

    # Theory-of-Mind reward
    bd["tom"] = 0.0
    if parsed and parsed.get("belief_agitation") is not None:
        try:
            from grader import compute_tom_reward
            h = st["env"]._hidden
            if h:
                tom = compute_tom_reward(
                    predicted_agitation=float(parsed.get("belief_agitation",5)),
                    actual_agitation=h.agitation,
                    predicted_demand=str(parsed.get("belief_demand","")),
                    actual_top_demand=h.demands[0].text if h.demands else "",
                    predicted_lying=bool(parsed.get("belief_lying",False)),
                    actually_lying=h.is_lying_about_hostages or h.is_lying_about_weapon,
                )
                bd["tom"] = round(tom, 4)
        except Exception: pass

    heuristic_score = max(-0.5, min(1.0, sum(bd.values()) + parse_bonus))
    bd["heuristic_score"] = round(heuristic_score, 4)

    # Multi-turn env trajectory (deepcopy)
    bd["env_traj"] = 0.0
    bd["outcome_bonus"] = 0.0
    try:
        env_copy = copy.deepcopy(st["env"])
        traj_r = 0.0
        step_obs = env_copy.step(action)
        traj_r += float(getattr(step_obs,"reward",0.0))
        for t in range(MULTI_TURN_STEPS - 1):
            if getattr(step_obs,"done",False): break
            at, cnt = HEURISTIC[t % len(HEURISTIC)]
            step_obs = env_copy.step(NegotiatorAction(action_type=at, content=cnt,
                                                       reasoning="traj", target="hostage_taker"))
            traj_r += float(getattr(step_obs,"reward",0.0))
        bd["env_traj"] = round(traj_r / MULTI_TURN_STEPS, 4)
        if getattr(step_obs,"done",False):
            msg = (getattr(step_obs,"message","") or "").lower()
            bd["outcome_bonus"] = 0.30 if any(k in msg for k in ["surrender","released"]) else \
                                  (-0.25 if any(k in msg for k in ["harm","tactical","supervisor"]) else 0.0)
        del env_copy
    except Exception as e:
        pass  # fallback: heuristic only

    blended = max(-0.5, min(1.0, 0.40*heuristic_score + 0.40*bd["env_traj"] + bd["outcome_bonus"]))
    return float(blended), bd


def advance_neg_env(prompt_idx: int, completion: str):
    st = _NEG_STATE.get(prompt_idx)
    if st is None or getattr(st.get("obs"),"done",False): return
    parsed = parse_action(completion)
    action, _ = to_action(parsed, completion)
    obs = st["env"].step(action)
    st["obs"] = obs
    st.setdefault("actions",[]).append(action.action_type)


def ht_reward_fn(completion: str) -> float:
    lo = completion.lower()
    s  = 0.0
    if any(w in lo for w in ["no","won't","never","don't trust","liar"]): s += 0.15
    if any(w in lo for w in ["i want","i need","give me","demand"]):      s += 0.10
    if any(w in lo for w in ["!","why","nobody","always"]):               s += 0.05
    if any(w in lo for w in ["i give up","surrender","you win"]):         s -= 0.30
    if any(w in lo for w in ["as an ai","language model"]):               s -= 0.20
    words = len(completion.split())
    s += 0.05 if 10 <= words <= 60 else (-0.10 if words > 120 else 0.0)
    return max(-0.5, min(0.5, s))

print("✓ Reward functions defined")


## 🤖 Cell 9 — Load Model with Unsloth (4-bit + LoRA)

In [ ]:
from unsloth import FastLanguageModel

def load_model_unsloth(adapter_path=None):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        dtype=None,           # auto-detect: bfloat16 on A100, float16 on T4
        load_in_4bit=True,    # 4-bit quantisation — fits on 15 GB T4
    )

    if adapter_path and __import__("pathlib").Path(adapter_path).exists():
        # Resume from checkpoint
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path, is_trainable=True)
        print(f"[model] Resumed from adapter: {adapter_path}")
    else:
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=["q_proj","k_proj","v_proj","o_proj"],
            lora_dropout=0.0,
            bias="none",
            use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient checkpointing
            random_state=SEED,
        )
        print("[model] Fresh LoRA initialised")

    model.print_trainable_parameters()
    return model, tokenizer

# Load once to verify
model, tokenizer = load_model_unsloth()
print("\n✓ Model loaded successfully")


## 🏋️ Cell 10 — Co-Evolution Training Loop

Alternates:
- **Even rounds** → train the **Negotiator** with GRPO (env-connected reward)
- **Odd rounds** → train the **Hostage-Taker** with GRPO (resistance reward)

Each round the canonical env is advanced with the best GRPO completion, so later steps see richer dialogue context.


In [ ]:
import gc, time
from pathlib import Path
from trl import GRPOConfig, GRPOTrainer

TRAIN_LOG = []  # global log for plotting

def _grpo_config(output_dir):
    return GRPOConfig(
        output_dir=output_dir,
        num_generations=NUM_GENERATIONS,
        generation_batch_size=NUM_GENERATIONS,
        max_completion_length=MAX_NEW_TOKENS,
        loss_type="dr_grpo",          # Dr. GRPO: removes 1/|o| length bias
        temperature=0.9,
        beta=0.04,
        per_device_train_batch_size=NUM_GENERATIONS,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        learning_rate=LEARNING_RATE,
        logging_steps=2,
        save_steps=9999,
        save_total_limit=1,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=False,
        report_to=[],
        remove_unused_columns=False,
    )

t_total = time.time()
print(f"=== Co-Evolution GRPO | {NUM_ROUNDS} rounds × {NUM_PROMPTS} prompts ===\n")

for rnd in range(NUM_ROUNDS):
    t_rnd = time.time()

    if rnd % 2 == 0:
        print(f"\n{'='*55}")
        print(f" Round {rnd+1}/{NUM_ROUNDS}: NEGOTIATOR")
        print(f"{'='*55}")

        adapter = NEG_OUTPUT if rnd > 0 and Path(NEG_OUTPUT).exists() else None
        model, tokenizer = load_model_unsloth(adapter)
        ds = build_neg_dataset(tokenizer, NUM_PROMPTS, current_round=rnd)

        step_count = [0]
        def neg_reward_fn(completions, prompt_idx=None, **kw):
            idxs = list(prompt_idx) if prompt_idx is not None else list(range(len(completions)))
            rewards, groups = [], {}
            for c, idx in zip(completions, idxs):
                score, bd = score_negotiator(int(idx), c)
                rewards.append(score)
                groups.setdefault(int(idx), []).append((c, score))
                step_count[0] += 1
                TRAIN_LOG.append({"step": step_count[0], "agent":"neg", "round":rnd, "reward":score,
                                  "heuristic": bd.get("heuristic_score",0),
                                  "env_traj": bd.get("env_traj",0),
                                  "outcome": bd.get("outcome_bonus",0)})
            for idx, items in groups.items():
                best_c, _ = max(items, key=lambda x: x[1])
                advance_neg_env(idx, best_c)
            mean_r = sum(rewards)/len(rewards)
            if step_count[0] % (NUM_GENERATIONS*4) == 0:
                print(f"  step={step_count[0]:4d} | mean_reward={mean_r:.4f}")
            return rewards

        trainer = GRPOTrainer(model=model, args=_grpo_config(NEG_OUTPUT),
                              reward_funcs=[neg_reward_fn], train_dataset=ds,
                              processing_class=tokenizer)
        trainer.train()
        model.save_pretrained(NEG_OUTPUT); tokenizer.save_pretrained(NEG_OUTPUT)
        print(f"  Saved → {NEG_OUTPUT}")

    else:
        print(f"\n{'='*55}")
        print(f" Round {rnd+1}/{NUM_ROUNDS}: HOSTAGE-TAKER")
        print(f"{'='*55}")

        adapter = HT_OUTPUT if rnd > 1 and Path(HT_OUTPUT).exists() else None
        model, tokenizer = load_model_unsloth(adapter)
        ds = build_ht_dataset(tokenizer, NUM_PROMPTS, current_round=rnd)

        ht_step = [0]
        def ht_reward_wrapper(completions, **kw):
            ht_step[0] += 1
            rewards = [ht_reward_fn(c) for c in completions]
            TRAIN_LOG.append({"step": ht_step[0], "agent":"ht", "round":rnd,
                              "reward": sum(rewards)/len(rewards)})
            return rewards

        trainer = GRPOTrainer(model=model, args=_grpo_config(HT_OUTPUT),
                              reward_funcs=[ht_reward_wrapper], train_dataset=ds,
                              processing_class=tokenizer)
        trainer.train()
        model.save_pretrained(HT_OUTPUT); tokenizer.save_pretrained(HT_OUTPUT)
        print(f"  Saved → {HT_OUTPUT}")

    # Free memory between rounds
    del model, tokenizer, trainer, ds
    gc.collect(); torch.cuda.empty_cache()
    print(f"  Round {rnd+1} done in {(time.time()-t_rnd)/60:.1f} min")

print(f"\n✓ Training complete in {(time.time()-t_total)/60:.1f} min")


## 📈 Cell 11 — Reward Curves

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Separate negotiator and HT logs
neg_log = [e for e in TRAIN_LOG if e["agent"] == "neg"]
ht_log  = [e for e in TRAIN_LOG if e["agent"] == "ht"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Crisis Negotiator — GRPO Training Curves", fontsize=14, fontweight="bold")

# ── Left: Negotiator reward components ───────────────────────────────────────
ax = axes[0]
if neg_log:
    steps    = [e["step"] for e in neg_log]
    rewards  = [e["reward"] for e in neg_log]
    heur     = [e.get("heuristic",0) for e in neg_log]
    env_traj = [e.get("env_traj",0) for e in neg_log]

    def smooth(xs, w=20):
        if len(xs) < w: return xs
        return np.convolve(xs, np.ones(w)/w, mode="valid").tolist()

    s_steps   = steps[len(steps)-len(smooth(rewards)):]
    ax.plot(s_steps, smooth(rewards),  label="Blended reward", color="#2196F3", lw=2)
    ax.plot(s_steps, smooth(heur),     label="Heuristic score", color="#4CAF50", lw=1.2, ls="--")
    ax.plot(s_steps, smooth(env_traj), label="Env trajectory",  color="#FF9800", lw=1.2, ls="--")
    ax.axhline(0, color="gray", lw=0.5, ls=":")
    # Round boundaries
    round_steps = {}
    for e in neg_log:
        round_steps.setdefault(e["round"], e["step"])
    for r, s in round_steps.items():
        ax.axvline(s, color="purple", lw=0.8, ls=":", alpha=0.6)
        ax.text(s, ax.get_ylim()[0]*0.95 if ax.get_ylim()[0] < 0 else 0.02,
                f"R{r+1}", color="purple", fontsize=8)
    ax.set_title("Negotiator Reward (smoothed, w=20)")
    ax.set_xlabel("Training step"); ax.set_ylabel("Reward")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Right: Reward distribution by component (final round) ────────────────────
ax = axes[1]
if neg_log:
    from collections import defaultdict
    last_round = max(e["round"] for e in neg_log)
    last_entries = [e for e in neg_log if e["round"] == last_round][-200:]
    r_vals = [e["reward"] for e in last_entries]
    ax.hist(r_vals, bins=30, color="#2196F3", alpha=0.75, edgecolor="white")
    ax.axvline(np.mean(r_vals), color="red", lw=2, label=f"Mean={np.mean(r_vals):.3f}")
    ax.axvline(np.median(r_vals), color="orange", lw=1.5, ls="--", label=f"Median={np.median(r_vals):.3f}")
    ax.set_title(f"Reward Distribution — Round {last_round+1} (last 200 steps)")
    ax.set_xlabel("Reward"); ax.set_ylabel("Count")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("reward_curve_coevolve_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved reward_curve_coevolve_v3.png")


## 🎯 Cell 12 — Evaluation vs Baselines

In [ ]:
import sys; sys.path.insert(0, "eval")
from eval_baselines import RandomPolicy, HeuristicPolicy, TrainedPolicy, run_episodes, summarize
import json

N_EVAL = 20  # episodes per policy (increase for more stable estimates)
DIFFS  = ["easy","medium","hard"]

print("=" * 55)
print(" Evaluating 3 policies × 3 difficulties × %d episodes" % N_EVAL)
print("=" * 55)

results = {}

print("\n[1/3] Random policy...")
results["random"] = summarize(run_episodes(RandomPolicy(), N_EVAL, DIFFS, seed_offset=9000))

print("[2/3] Heuristic policy (BCSM cycle)...")
results["heuristic"] = summarize(run_episodes(HeuristicPolicy(), N_EVAL, DIFFS, seed_offset=9000))

print("[3/3] Trained policy (LoRA)...")
try:
    trained = TrainedPolicy(NEG_OUTPUT)
    results["trained"] = summarize(run_episodes(trained, N_EVAL, DIFFS, seed_offset=9000))
except Exception as e:
    print(f"  Could not load trained model: {e}")
    results["trained"] = None

# Print summary table
print("\n" + "=" * 55)
print(f"{'Policy':<12} {'Reward':>8} {'Surrender%':>11} {'Harm%':>7} {'Steps':>6}")
print("-" * 55)
for name, s in results.items():
    if s is None:
        print(f"{name:<12}  (not available)")
        continue
    print(f"{name:<12} {s.get('mean_reward',0):>8.4f} "
          f"{s.get('surrender_rate',0)*100:>10.1f}% "
          f"{s.get('harm_rate',0)*100:>6.1f}% "
          f"{s.get('mean_steps',0):>6.1f}")
print("=" * 55)

with open("eval_summary_coevolve_v3.json","w") as f:
    json.dump(results, f, indent=2)
print("\n✓ Saved eval_summary_coevolve_v3.json")


## 💾 Cell 13 — Save / Push to Hugging Face Hub (Optional)

In [ ]:
import shutil, pathlib, os

# Zip the adapter for download
adapter_dir = pathlib.Path(NEG_OUTPUT)
if adapter_dir.exists():
    shutil.make_archive("crisis-negotiator-grpo", "zip", adapter_dir)
    print("✓ Adapter zipped → crisis-negotiator-grpo.zip")

# ── Optional: push to HF Hub ─────────────────────────────────────────────────
# Uncomment and fill in your HF_TOKEN + repo name to push
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=NEG_OUTPUT,
#     repo_id="YOUR_HF_USERNAME/crisis-negotiator-3b-grpo",
#     repo_type="model",
#     token="YOUR_HF_TOKEN",
# )
# print("Pushed to HF Hub")

print("\n=== Training complete! ===")
print(f"Negotiator adapter : {NEG_OUTPUT}/")
print(f"Reward plot        : reward_curve_coevolve_v3.png")
print(f"Eval summary       : eval_summary_coevolve_v3.json")
